# Autoship Nudge Promo Incentive — Power Analysis (Autoship Adoption Rate, 2-Cell Design, Verified Population)

**Experiment:** Autoship Nudge Promo Incentive Test · **Owner:** Sergio Oyola · **Primary metric analyzed here:** Autoship Adoption Rate · **Randomization unit:** `client_id` · **Allocation point:** post-First-Fix checkout, once keep rate is known, at the moment the client selects a Quick Fix date and clicks "Schedule a Quick Fix" (prior to Narvar handoff)

This notebook sizes a standard 2-cell A/B version of the experiment for its primary metric, **Autoship Adoption Rate**, adding three population-integrity checks on top of the base eligibility criteria, with the adoption rate and eligible daily volume both pooled across a stable multi-month window.

## Population
Manual clients — i.e., not already enrolled in Autoship — who completed First Fix checkout with a **Buy 1+** keep rate (kept at least one item). Buy 0 clients always see the BAU Quick Fix experience with no Autoship nudge and are out of scope for this comparison, per the PRD. On top of these base criteria, three additional checks are applied:

1. **Genuinely first-time, not reactivated.** Cross-checked against `curated.checkout_based_client_state_journal` as of First Fix checkout; only `'Never Active'` clients qualify — not `Engaged`, `Dormant`, or `Lapsed`.
2. **Employees excluded.** `curated.client.employee_affiliated_flag` marks employee-affiliated accounts, which are excluded from the eligible population.
3. **Fraudulent clients excluded.** `curated.client.fake_client_flag` marks accounts identified as fraudulent, which are excluded from the eligible population.

Separately, both the adoption-rate baseline and eligible daily volume (Step 1c) are confirmed against the observed share of this population who actually complete the "Schedule a Quick Fix" click on record in `curated.product_tracking_events` — that click is what actually triggers randomization, so qualifying on the criteria above is necessary but not sufficient to be allocated.

## Design: 2-cell test, single comparison
Eligible clients are randomized into 2 cells at a 50/50 split:

| Cell | Experience | Offer |
|---|---|---|
| Control | BAU Quick Fix, no Autoship nudge | None |
| Treatment | Autoship nudge + promo billboard | 10% off next eligible Fix |

A single pairwise comparison is planned: **Treatment vs. Control**, measuring the combined effect of introducing the Autoship nudge together with the promo incentive, relative to today's BAU experience. Because only one comparison is planned against the family-wise error budget, no multiple-comparison correction is needed here — sizing uses the initial `alpha = 0.05` directly.

## One-sided test
A flat result and a negative result both lead to the same rollout decision — don't ship the nudge+promo experience — so there's no need to distinguish "no effect" from "harmful." This sizing is **one-sided**, powered only to detect a positive lift.

## Metric definition
**Autoship Adoption Rate** = share of eligible clients who show a fresh Autoship demand event within a 90-day window following their First Fix checkout.

A client's Autoship history is tracked in `curated.client_pulse_journal`, a daily journal (one row per day any tracked client attribute changes) carrying `last_autoship_demand_ts` — the timestamp of that client's most recent Autoship demand event as of that journal row. A client is counted as **adopted** if, scanning their full journal history, the *earliest* `last_autoship_demand_ts` value that is itself later than their First Fix checkout date falls within 90 days of that checkout.

- **Full journal history is scanned, not a single snapshot** — `last_autoship_demand_ts` resets on full cancellation, so only the earliest post-First-Fix value is robust to that.
- **The demand timestamp must be strictly *after* First Fix checkout** — excludes clients whose Autoship timestamp predates First Fix entirely (a pre-existing enrollment unrelated to this nudge).

**Caveat:** this is an opt-in-*adjacent* signal (subscription creation, not a literal click event) and can't confirm the post-First-Fix nudge specifically caused the adoption.

In [1]:
import numpy as np
import pandas as pd
from amphibian import get_data_accessor
from power import n_total_statsmodels

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

def query(sql):
    return get_data_accessor(engines=['presto']).fetch_sql(sql=sql)

# Data parameters
COHORT_START = '2025-08-01'
STABLE_WINDOW_END = '2026-06-01'  # exclusive upper bound: months on/after this date accelerate sharply with no counterpart in the live experiment's own allocation rate
MATURATION_DAYS = 90  # days to wait for a client's Autoship demand event to resolve

# Design parameters (2-cell test, single comparison, no multiple-comparison correction)
ALPHA = 0.05  # single comparison: Treatment vs. Control, no Bonferroni adjustment needed
POWER = 0.80
TWO_SIDED = False  # one-sided: only a positive lift over BAU changes the rollout decision
N_ARMS = 2
SPLIT = 0.5  # Control and Treatment are equal-sized arms
MDE_GRID = [0.02, 0.03, 0.04, 0.05, 0.10, 0.12, 0.15]  # relative lift on Autoship Adoption Rate, Treatment vs. Control

## Step 1 — Autoship Adoption Rate baseline & daily eligible volume

Adoption rate and daily volume are pooled in one query across `COHORT_START` through `STABLE_WINDOW_END`: within that window, monthly volume moves within a stable range with no persistent trend, so pooling gives a more robust estimate than any single month. `STABLE_WINDOW_END` sits well past `MATURATION_DAYS`, so every checkout here has had time to fully resolve.

In [2]:
baseline_volume_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
first_fix_deduped AS (
    SELECT client_id, checkout_date, autoship_or_manual, n_items_kept
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
),
state_at_fix AS (
    -- Each client's state as of the journal row covering their First Fix checkout.
    SELECT f.client_id, j.client_state_detail,
           ROW_NUMBER() OVER (PARTITION BY f.client_id ORDER BY j.start_timestamp DESC) AS rn
    FROM first_fix_deduped f
    JOIN curated.checkout_based_client_state_journal j
      ON j.client_id = f.client_id
     AND j.start_timestamp <= CAST(f.checkout_date AS TIMESTAMP)
),
eligible AS (
    SELECT f.client_id, f.checkout_date
    FROM first_fix_deduped f
    JOIN curated.client c ON c.client_id = f.client_id
    JOIN state_at_fix s ON s.client_id = f.client_id AND s.rn = 1
    WHERE f.autoship_or_manual = 'manual'
      AND f.n_items_kept >= 1
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND s.client_state_detail = 'Never Active'
      AND f.checkout_date >= DATE '{COHORT_START}'
      AND f.checkout_date <  DATE '{STABLE_WINDOW_END}'
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
)
SELECT
    COUNT(DISTINCT e.client_id) AS n_eligible,
    COUNT(DISTINCT CASE WHEN f.first_fresh_demand_ts IS NOT NULL
                         AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
                        THEN e.client_id END) AS n_adopted,
    COUNT(DISTINCT DATE_TRUNC('day', e.checkout_date)) AS days_observed
FROM eligible e
LEFT JOIN fresh_demand f ON f.client_id = e.client_id
"""

baseline_volume_df = query(baseline_volume_query)
baseline_volume_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,n_eligible,n_adopted,days_observed
0,86893,19305,304


In [3]:
BASELINE_RATE = float(baseline_volume_df['n_adopted'][0]) / float(baseline_volume_df['n_eligible'][0])
DAILY_ELIGIBLE = float(baseline_volume_df['n_eligible'][0]) / float(baseline_volume_df['days_observed'][0])

print(f"BASELINE_RATE (pooled, {COHORT_START} to {STABLE_WINDOW_END}) = {BASELINE_RATE:.4f}  |  DAILY_ELIGIBLE = {DAILY_ELIGIBLE:,.1f} / day")

BASELINE_RATE (pooled, 2025-08-01 to 2026-06-01) = 0.2222  |  DAILY_ELIGIBLE = 285.8 / day


## Step 1b — Why the most recent months are excluded

This population's monthly split between Manual and Autoship First Fixes is checked directly, over a longer span than the pooled window above, to confirm where and why it stops looking stable.

In [4]:
validation_query = f"""--sql
SELECT
    DATE_TRUNC('month', checkout_date) AS checkout_month,
    autoship_or_manual,
    COUNT(DISTINCT client_id) AS n_clients
FROM curated.merch_sales_and_feedback
WHERE fix_number = 1
  AND created_date >= DATE '{COHORT_START}'
  AND checkout_date >= DATE '{COHORT_START}'
GROUP BY 1, 2
ORDER BY 1, 2
"""

validation_df = query(validation_query)
validation_pivot = validation_df.pivot(index='checkout_month', columns='autoship_or_manual', values='n_clients')
validation_pivot['total'] = validation_pivot.sum(axis=1)
validation_pivot['manual_share'] = (validation_pivot['manual'] / validation_pivot['total']).round(3)
validation_pivot

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


autoship_or_manual,autoship,manual,total,manual_share
checkout_month,,,,
2025-08-01,31088,9095,40183,0.226
2025-09-01,45588,12125,57713,0.210
2025-10-01,44237,12041,56278,0.214
2025-11-01,35767,9681,45448,0.213
2025-12-01,24583,11842,36425,0.325
2026-01-01,43651,14420,58071,0.248
2026-02-01,42628,12289,54917,0.224
2026-03-01,53660,15076,68736,0.219
2026-04-01,44781,13974,58755,0.238


**Reading this:** total First Fix volume (manual + Autoship combined) stays in a comparatively narrow band throughout, but the manual share holds roughly steady through the pooled window above and then climbs sharply in the months right before today. That's a structural shift in this population, not noise, so the pooled window above is drawn only from the months before that shift.

## Step 1c — Confirming reach via the "Schedule a Quick Fix" click

Qualifying isn't the same as reaching the allocation trigger: a client is only randomized once they click "Schedule a Quick Fix." `curated.product_tracking_events` records this exact click (`name = 'schedule_quick_fix_button'`, `action_name = 'schedule_quick_fix'`, `post_checkout_promo` screen), measured over the same pooled window as Step 1.

In [5]:
reach_query = f"""--sql
WITH first_fix AS (
    SELECT
        client_id,
        shipment_id,
        MIN(checkout_date) AS checkout_date,
        ARBITRARY(autoship_or_manual) AS autoship_or_manual,
        SUM(sold_paid_fix_flag) AS n_items_kept
    FROM curated.merch_sales_and_feedback
    WHERE fix_number = 1
      AND created_date >= DATE '{COHORT_START}'
    GROUP BY client_id, shipment_id
),
first_fix_deduped AS (
    SELECT client_id, checkout_date, autoship_or_manual, n_items_kept
    FROM (
        SELECT client_id, checkout_date, autoship_or_manual, n_items_kept,
               ROW_NUMBER() OVER (PARTITION BY client_id ORDER BY checkout_date) AS rn
        FROM first_fix
    )
    WHERE rn = 1
),
state_at_fix AS (
    SELECT f.client_id, j.client_state_detail,
           ROW_NUMBER() OVER (PARTITION BY f.client_id ORDER BY j.start_timestamp DESC) AS rn
    FROM first_fix_deduped f
    JOIN curated.checkout_based_client_state_journal j
      ON j.client_id = f.client_id
     AND j.start_timestamp <= CAST(f.checkout_date AS TIMESTAMP)
),
eligible AS (
    SELECT f.client_id, f.checkout_date
    FROM first_fix_deduped f
    JOIN curated.client c ON c.client_id = f.client_id
    JOIN state_at_fix s ON s.client_id = f.client_id AND s.rn = 1
    WHERE f.autoship_or_manual = 'manual'
      AND f.n_items_kept >= 1
      AND COALESCE(c.fake_client_flag, 0) = 0
      AND COALESCE(c.employee_affiliated_flag, 0) = 0
      AND s.client_state_detail = 'Never Active'
      AND f.checkout_date >= DATE '{COHORT_START}'
      AND f.checkout_date <  DATE '{STABLE_WINDOW_END}'
),
reached AS (
    SELECT DISTINCT e.client_id
    FROM eligible e
    JOIN curated.product_tracking_events t
      ON t.client_id = e.client_id
     AND t.name = 'schedule_quick_fix_button'
     AND t.action_name = 'schedule_quick_fix'
     AND t.screen_view_name = 'post_checkout_promo'
     AND t.event_timestamp >= CAST(e.checkout_date AS TIMESTAMP)
    WHERE t.date_in_utc >= DATE '{COHORT_START}'
),
fresh_demand AS (
    SELECT e.client_id, MIN(p.last_autoship_demand_ts) AS first_fresh_demand_ts
    FROM eligible e
    JOIN curated.client_pulse_journal p
      ON p.client_id = e.client_id
     AND p.last_autoship_demand_ts > e.checkout_date
    GROUP BY e.client_id
)
SELECT
    COUNT(DISTINCT e.client_id) AS n_eligible,
    COUNT(DISTINCT r.client_id) AS n_reached,
    COUNT(DISTINCT CASE WHEN r.client_id IS NOT NULL AND f.first_fresh_demand_ts IS NOT NULL
                         AND f.first_fresh_demand_ts <= e.checkout_date + INTERVAL '{MATURATION_DAYS}' DAY
                        THEN e.client_id END) AS n_adopted_reached,
    COUNT(DISTINCT DATE_TRUNC('day', e.checkout_date)) AS days_observed
FROM eligible e
LEFT JOIN reached r ON r.client_id = e.client_id
LEFT JOIN fresh_demand f ON f.client_id = e.client_id
"""

reach_df = query(reach_query)
REACH_RATE = float(reach_df['n_reached'][0]) / float(reach_df['n_eligible'][0])
BASELINE_RATE_REACHED = float(reach_df['n_adopted_reached'][0]) / float(reach_df['n_reached'][0])
DAILY_ELIGIBLE_REACHED = float(reach_df['n_reached'][0]) / float(reach_df['days_observed'][0])

print(f"REACH_RATE = {REACH_RATE:.1%}  |  BASELINE_RATE_REACHED = {BASELINE_RATE_REACHED:.4f}  |  DAILY_ELIGIBLE_REACHED = {DAILY_ELIGIBLE_REACHED:,.1f} / day")
reach_df

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


REACH_RATE = 30.6%  |  BASELINE_RATE_REACHED = 0.3361  |  DAILY_ELIGIBLE_REACHED = 87.5 / day


,n_eligible,n_reached,n_adopted_reached,days_observed
0,86892,26599,8940,304


**Reading this:** a meaningful share of the eligible population never completes the "Schedule a Quick Fix" click on record. `BASELINE_RATE_REACHED` and `DAILY_ELIGIBLE_REACHED` — the rate and volume measured specifically among clients confirmed to have reached that click — are what the sizing below uses.

## Step 2 — Sample size & duration

`n_total_statsmodels` sizes a single pairwise 50/50 comparison (Treatment vs. Control); `n_treatment` is read as the **per-arm** requirement. Each of the 2 arms accrues `DAILY_ELIGIBLE_REACHED / 2` reached clients per day under the 50/50 split, so `days_required = n_per_arm / (DAILY_ELIGIBLE_REACHED / 2)`.

In [6]:
def size_table(rel_grid, baseline, daily):
    raw = n_total_statsmodels(
        baseline_rate=baseline, mde_relative=rel_grid, split_ratio=[SPLIT],
        alpha=ALPHA, power=POWER, two_sided=TWO_SIDED,
    )
    df = pd.DataFrame(raw).T.reset_index(drop=True)
    df['rel_effect'] = df['mde_relative'].apply(lambda x: f"{x:+.0%}")
    df['n_per_arm'] = df['n_treatment'].astype(int)
    df['n_total_2arm'] = df['n_per_arm'] * N_ARMS
    df['days_required'] = np.ceil(df['n_per_arm'] / (daily / N_ARMS)).astype(int)
    df['weeks_required'] = (df['days_required'] / 7).round(1)
    return df[['rel_effect', 'p_treatment', 'n_per_arm', 'n_total_2arm', 'days_required', 'weeks_required']]

sided = 'one-sided' if not TWO_SIDED else 'two-sided'
print(f"--- Autoship Adoption Rate, Treatment vs. Control (baseline={BASELINE_RATE_REACHED:.1%}, alpha={ALPHA}, power={POWER:.0%}, {sided}, 50/50 split) ---")
size_table(MDE_GRID, BASELINE_RATE_REACHED, DAILY_ELIGIBLE_REACHED)

--- Autoship Adoption Rate, Treatment vs. Control (baseline=33.6%, alpha=0.05, power=80%, one-sided, 50/50 split) ---


,rel_effect,p_treatment,n_per_arm,n_total_2arm,days_required,weeks_required
0,+2%,0.342825,61359,122718,1403,200.4
1,+3%,0.346186,27336,54672,625,89.3
2,+4%,0.349547,15412,30824,353,50.4
3,+5%,0.352908,9887,19774,226,32.3
4,+10%,0.369713,2499,4998,58,8.3
5,+12%,0.376435,1743,3486,40,5.7
6,+15%,0.386518,1122,2244,26,3.7


**Reading this:** runtime drops sharply at larger MDEs but stays long at single-digit ones; a 2-cell design still runs shorter than a 3-cell one at any MDE — no Bonferroni correction (alpha 0.05 not 0.025) and each arm gets half the daily traffic instead of a third. No harm/guardrail grid here: this sizes a one-sided positive MDE only; margin risk is covered qualitatively elsewhere (Experiment Design doc).

## Step 3 — Summary for the Experiment Design doc

Headline MDE below is a **placeholder 12% relative lift** on Autoship Adoption Rate (Treatment vs. Control) — the second-largest value in the Step 2 grid.

In [7]:
TARGET_REL_MDE = 0.12  # placeholder: 12% relative lift on Autoship Adoption Rate, Treatment vs. Control (second-largest value in the Step 2 grid)

res = n_total_statsmodels(
    baseline_rate=BASELINE_RATE_REACHED,
    mde_relative=[TARGET_REL_MDE],
    split_ratio=[SPLIT],
    alpha=ALPHA,
    power=POWER,
    two_sided=TWO_SIDED,
)

n_per_arm = int(list(res.values())[0]['n_treatment'])
duration_days = int(np.ceil(n_per_arm / (DAILY_ELIGIBLE_REACHED / N_ARMS)))

summary = {
    'Metric Used': 'Autoship Adoption Rate (Treatment vs. Control)',
    'Population': 'Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients',
    'Baseline Value': f"{BASELINE_RATE_REACHED:.1%} (pooled {COHORT_START} to {STABLE_WINDOW_END}, {MATURATION_DAYS}-day matured, among clients confirmed to have reached the click)",
    'Daily Eligible Volume': f"{DAILY_ELIGIBLE_REACHED:,.1f} / day (pooled {COHORT_START} to {STABLE_WINDOW_END}, among clients confirmed to have reached the click, {REACH_RATE:.0%} observed reach rate)",
    'Minimum Detectable Effect': f"+{TARGET_REL_MDE:.0%} relative ({BASELINE_RATE_REACHED:.3f} -> {BASELINE_RATE_REACHED*(1+TARGET_REL_MDE):.3f})",
    'One/Two-Sided Test': 'One-sided',
    'Significance Level': f"{ALPHA} (single comparison, no multiple-comparison correction)",
    'Statistical Power': f"{POWER:.0%}",
    'Variant Split %': '50% / 50% (Control / Treatment)',
    'Minimum Samples by Variant': f"{n_per_arm:,}",
    'Minimum Samples total (2 arms)': f"{n_per_arm*N_ARMS:,}",
    'Shortest Duration Required': f"{duration_days} days (~{duration_days/7:.1f} weeks)",
}
pd.Series(summary).to_frame('value')

,value
Metric Used,Autoship Adoption Rate (Treatment vs. Control)
Population,"Manual clients, First Fix checkout complete, Buy 1+ keep rate, not already enrolled in Autoship, verified Never Active prior to First Fix, excluding employees and fraudulent clients"
Baseline Value,"33.6% (pooled 2025-08-01 to 2026-06-01, 90-day matured, among clients confirmed to have reached the click)"
Daily Eligible Volume,"87.5 / day (pooled 2025-08-01 to 2026-06-01, among clients confirmed to have reached the click, 31% observed reach rate)"
Minimum Detectable Effect,+12% relative (0.336 -> 0.376)
One/Two-Sided Test,One-sided
Significance Level,"0.05 (single comparison, no multiple-comparison correction)"
Statistical Power,80%
Variant Split %,50% / 50% (Control / Treatment)
Minimum Samples by Variant,"1,743"


## Bottom line

- **Three integrity checks on top of base eligibility:** `'Never Active'` at First Fix (excludes reactivated clients), no employees, no fraud.
- **Baseline rate and daily volume are pooled across a stable window** (`COHORT_START`-`STABLE_WINDOW_END`), a more robust estimate than any single month.
- **The window stops before a sharp, unexplained acceleration** in manual share (Step 1b) that has no counterpart in the live experiment's own flat allocation rate.
- **Baseline captures fresh post-First-Fix Autoship demand within 90 days**, robust to later cancellation, and excludes pre-existing Autoship enrollment unrelated to this nudge.
- **Both baseline and volume are confirmed against the "Schedule a Quick Fix" click** (Step 1c) — `BASELINE_RATE_REACHED` / `DAILY_ELIGIBLE_REACHED` are what this notebook sizes against.
- **2-cell design removes Bonferroni** (alpha 0.05, not 0.025) and gives each arm half the daily traffic instead of a third — both shorten runtime.
- At **12% relative lift** (placeholder MDE), sample size and duration are in Step 3; smaller MDEs run considerably longer, 15% is faster (Step 2).